# Model Selection 

This notebook compares multiple modeling and encoding strategies for predicting Gurgaon property prices. It builds on the cleaned and feature-engineered dataset to benchmark several baselines, tune key hyperparameters, and select a strong model for deployment.

In [1]:
from config import DATA_PROCESSED, MODELS_DIR
import sys
print(sys.executable)

d:\Coding\Projects\Habitalytics\notebooks_venv\Scripts\python.exe


In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, r2_score, make_scorer
from sklearn.decomposition import PCA

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

In [3]:
df = pd.read_csv(DATA_PROCESSED / 'gurgaon_properties_post_feature_selection_v2.csv')

In [4]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 36,0.82,3.0,2.0,2,New Property,850.0,0.0,0.0,0.0,Low,Low Floor
1,flat,sector 89,0.95,2.0,2.0,2,New Property,1226.0,1.0,0.0,0.0,Low,Mid Floor
2,flat,sohna road,0.32,2.0,2.0,1,New Property,1000.0,0.0,0.0,0.0,Low,High Floor
3,flat,sector 92,1.60,3.0,4.0,3+,Relatively New,1615.0,1.0,0.0,1.0,High,Mid Floor
4,flat,sector 102,0.48,2.0,2.0,1,Relatively New,582.0,0.0,1.0,0.0,High,Mid Floor


#### Converting funishing_type back to categorical

In [5]:
df['furnishing_type'].value_counts()

furnishing_type
0.0    2349
1.0    1018
2.0     187
Name: count, dtype: int64

In [6]:
# 0 -> unfurnished
# 1 -> semifurnished
# 2 -> furnished
df['furnishing_type'] = df['furnishing_type'].replace({0.0:'unfurnished',1.0:'semifurnished',2.0:'furnished'})

In [7]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 36,0.82,3.0,2.0,2,New Property,850.0,0.0,0.0,unfurnished,Low,Low Floor
1,flat,sector 89,0.95,2.0,2.0,2,New Property,1226.0,1.0,0.0,unfurnished,Low,Mid Floor
2,flat,sohna road,0.32,2.0,2.0,1,New Property,1000.0,0.0,0.0,unfurnished,Low,High Floor
3,flat,sector 92,1.60,3.0,4.0,3+,Relatively New,1615.0,1.0,0.0,semifurnished,High,Mid Floor
4,flat,sector 102,0.48,2.0,2.0,1,Relatively New,582.0,0.0,1.0,unfurnished,High,Mid Floor


In [8]:
X = df.drop(columns=['price'])
y = df['price']

In [9]:
# Applying the log1p transformation to the target variable
y_transformed = np.log1p(y)

## Ordinal Encoding

In this section we apply an `OrdinalEncoder` to the selected categorical features and feed the resulting integer codes into a simple linear regression model. This provides a fast baseline for how well a purely linear model can perform when categorical variables are represented by ordered labels.

In [10]:
columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

In [11]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode)
    ], 
    remainder='passthrough'
)

In [12]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [13]:
# Custom scorer: R2 on original (price) scale for interpretability
def _r2_original_scale(y_true_log, y_pred_log):
    return r2_score(np.expm1(y_true_log), np.expm1(y_pred_log))

r2_original_scorer = make_scorer(_r2_original_scale)

In [14]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring=r2_original_scorer)

In [15]:
# R2 score and standard deviation
scores.mean(), scores.std()

(np.float64(0.48923553862936436), np.float64(0.14872042695273935))

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [17]:
pipeline.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [18]:
y_pred = pipeline.predict(X_test)

In [19]:
y_pred = np.expm1(y_pred)

In [20]:
# MAE
mean_absolute_error(np.expm1(y_test),y_pred)

0.9463822160089356

#### Convert this flow into a function:

In [21]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation (R2 on original price scale)
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring=r2_original_scorer)
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [22]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [23]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [24]:
model_output

[['linear_reg', np.float64(0.48923553862936436), 0.9463822160089356],
 ['svr', np.float64(0.6042932954390542), 0.8472636473483922],
 ['ridge', np.float64(0.48955993398013825), 0.9463387741853386],
 ['LASSO', np.float64(-0.010576910409314222), 1.528905986892753],
 ['decision tree', np.float64(0.6373828730479572), 0.7350475183864225],
 ['random forest', np.float64(0.8127742009244845), 0.5234949136971796],
 ['extra trees', np.float64(0.7833249260603292), 0.5472733246754603],
 ['gradient boosting', np.float64(0.8128032826045406), 0.5767307332381497],
 ['adaboost', np.float64(0.6972557391762637), 0.8618743341315243],
 ['mlp', np.float64(0.7356265664272804), 0.7021660271888501],
 ['xgboost', np.float64(0.8163451944336215), 0.5040475141482346]]

In [25]:
# Create a dataframe
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.816345,0.504048
5,random forest,0.812774,0.523495
6,extra trees,0.783325,0.547273
7,gradient boosting,0.812803,0.576731
9,mlp,0.735627,0.702166
4,decision tree,0.637383,0.735048
1,svr,0.604293,0.847264
8,adaboost,0.697256,0.861874
2,ridge,0.489560,0.946339
0,linear_reg,0.489236,0.946382


<b>Conclusion: Best scores</b> <br>
- xgboost - R2 (0.81), MAE (0.50)</b> 
- random forest	 - R2 (0.81), MAE (0.52)

## One-Hot Encoding

Here we replace the ordinal representation with a `OneHotEncoder` for the selected categorical features, allowing the model to learn separate effects for each category. We keep a linear regressor on top to isolate the impact of the encoding choice on predictive performance.

In [26]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first'),['sector','agePossession','furnishing_type'])
    ], 
    remainder='passthrough'
)

In [27]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [28]:
# K-fold cross-validation (R2 on original price scale)
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring=r2_original_scorer)

In [29]:
# R2 Score and Standard Deviation
scores.mean(), scores.std()

(np.float64(0.6541140767085698), np.float64(0.16037625290226204))

In [30]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [31]:
pipeline.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [32]:
y_pred = pipeline.predict(X_test)

In [33]:
y_pred = np.expm1(y_pred)

In [34]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.6497514315131458

In [35]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation (R2 on original price scale)
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring=r2_original_scorer)
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [36]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [37]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [38]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.826459,0.468523
10,xgboost,0.826449,0.493456
5,random forest,0.835746,0.496704
9,mlp,0.800597,0.521921
7,gradient boosting,0.824264,0.569390
0,linear_reg,0.654114,0.649751
2,ridge,0.652429,0.652915
4,decision tree,0.675360,0.693240
1,svr,0.611209,0.834124
8,adaboost,0.695303,0.836049


<b>Conclusion:</b> <br>
- Linear model performance improved, but tree models are stil giving best results. <br>
- Best score: extra trees - R2(0.83), MAE(0.47)

## Target Encoding

Target (mean) encoding for high-cardinality categorical variables.

In [39]:
import category_encoders as ce

columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [40]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [41]:
# K-fold cross-validation (R2 on original price scale)
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring=r2_original_scorer)

In [42]:
scores.mean(),scores.std()

(np.float64(0.6152292617425735), np.float64(0.20248854231006244))

In [43]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation (R2 on original price scale)
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring=r2_original_scorer)
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [44]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [45]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [46]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [47]:
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.832472,0.447518
6,extra trees,0.832398,0.450937
5,random forest,0.832416,0.452632
7,gradient boosting,0.826605,0.507777
4,decision tree,0.705196,0.571366
9,mlp,0.802131,0.659602
8,adaboost,0.755145,0.691327
0,linear_reg,0.615229,0.713011
2,ridge,0.615515,0.713523
1,svr,0.627235,0.818851


<b>Conclusion:</b> <br>
Best score: <br>
- xgboost - R2(0.83), MAE(0.44)
- random forest - R2(0.83), MAE(0.45)

## Hyperparameter Tuning on Random Forest

After identifying promising preprocessing choices, we tune key hyperparameters of a `RandomForestRegressor` (such as depth, number of trees, and feature subsampling) using cross-validation. This section searches over a grid of configurations to find a strong baseline tree ensemble for price prediction.

In [48]:
from sklearn.model_selection import RandomizedSearchCV

In [49]:
param_grid = {
    'regressor__n_estimators': [200, 300, 400, 500],
    'regressor__max_depth': [15, 20, 25, 30],
    'regressor__min_samples_split': [2, 5, 10, 15],
    'regressor__min_samples_leaf': [1, 2, 4, 6],
    'regressor__max_samples': [0.4, 0.5, 0.6, 0.7],
    'regressor__max_features': [None, 'sqrt', 0.6, 0.8],
    'regressor__bootstrap': [True],
    'regressor__min_impurity_decrease': [0.0, 0.0001, 0.001]
}

In [50]:
columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [51]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

In [52]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

In [53]:
#search = GridSearchCV(pipeline, param_grid, cv=kfold, scoring=r2_original_scorer, n_jobs=-1, verbose=4)
search = RandomizedSearchCV(
    pipeline, param_grid, n_iter=600, cv=kfold,
    scoring=r2_original_scorer, n_jobs=-1, random_state=42, verbose=4
)

In [54]:
search.fit(X, y_transformed)

Fitting 10 folds for each of 600 candidates, totalling 6000 fits


,estimator,Pipeline(step...m_state=42))])
,param_distributions,"{'regressor__bootstrap': [True], 'regressor__max_depth': [15, 20, ...], 'regressor__max_features': [None, 'sqrt', ...], 'regressor__max_samples': [0.4, 0.5, ...], ...}"
,n_iter,600
,scoring,make_scorer(_...hod='predict')
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,4
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [55]:
final_pipe = search.best_estimator_

In [56]:
search.best_params_

{'regressor__n_estimators': 200,
 'regressor__min_samples_split': 2,
 'regressor__min_samples_leaf': 1,
 'regressor__min_impurity_decrease': 0.0,
 'regressor__max_samples': 0.6,
 'regressor__max_features': 0.8,
 'regressor__max_depth': 25,
 'regressor__bootstrap': True}

In [57]:
search.best_score_

np.float64(0.8375812808806795)

In [58]:
# Holdout evaluation on original price scale
best_pipeline = search.best_estimator_
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(X, y_transformed, test_size=0.2, random_state=42)
best_pipeline.fit(X_train_rf, y_train_rf)
y_pred_log = best_pipeline.predict(X_test_rf)
y_true = np.expm1(y_test_rf)
y_pred = np.expm1(y_pred_log)
mae_score = mean_absolute_error(y_true, y_pred)
r2_score_val = r2_score(y_true, y_pred)

print(f"R2 (original scale): {r2_score_val:.4f}")
print(f"MAE (original scale, Crores): {mae_score:.4f}")

R2 (original scale): 0.8684
MAE (original scale, Crores): 0.4496


## Exporting the Selected Model

Finally, we fit the best-performing model on the full training data and persist the trained pipeline to disk (e.g., via `joblib`). This exported artifact can then be loaded by the web application for serving real-time price predictions.

In [59]:
final_pipe.fit(X,y_transformed)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [60]:
import joblib

joblib.dump(
    final_pipe,
    MODELS_DIR / "pipeline.joblib",
)

['D:\\Coding\\Projects\\Habitalytics\\models\\pipeline.joblib']

In [61]:
# Export input data X
import pickle
with open(MODELS_DIR / 'df.pkl', 'wb') as file:
    pickle.dump(X, file)